<a href="https://www.kaggle.com/code/rahman4li/an-analysis-google-translate-vs-chatgpt?scriptVersionId=340184374" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Did ChatGPT Replace Google Translate? A Decade of Search Attention

**TL;DR:** Search interest for ChatGPT first overtook Google Translate worldwide in **February 2024** - about 15 months after ChatGPT's public launch - and the gap has widened substantially since, though not in a perfectly straight line.

This notebook analyzes 10+ years of Google Trends data for **Google Translate**, **ChatGPT**, and **DeepL** to test a simple question: is generative AI *substituting for* dedicated translation tools, or just *complementing* them?

 Full write-up with methodology & regression tables: see the linked GitHub repo below.
 Dataset: you're looking at it - upvote if this is useful to you!

---


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11

# Works both on Kaggle and locally
KAGGLE_PATH = Path("/kaggle/input/analyzing-google-translate-vs-chatgpt")
DATA_DIR = KAGGLE_PATH if KAGGLE_PATH.exists() else Path(".")

CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")

## 1. Load the data

Three files:
- `multiTimeline_2016.csv` - monthly Google Trends index, Jan 2016 to present
- `multiTimeline_5_years.csv` - weekly Google Trends index, more granular recent view
- `data_milestones_manual.csv` - hand-compiled product launch/event timeline

Trends values are **relative** search interest (0-100 within each file's own time range), not absolute search volume.

In [ ]:
DATA_DIR = Path("/kaggle/input/datasets/rahman4li/analyzing-google-translate-vs-chatgpt")


monthly = pd.read_csv(DATA_DIR / "multiTimeline_2016.csv", parse_dates=["Month"])
weekly = pd.read_csv(DATA_DIR / "multiTimeline_5_years.csv", parse_dates=["Week"])
milestones = pd.read_csv(DATA_DIR / "data_milestones_manual.csv", parse_dates=["date"])

monthly.head()

## 2. The headline chart: search interest over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

colors = {"Google Translate": "#C89B3C", "ChatGPT": "#3F8C79", "DeepL": "#8A8370"}
for col, color in colors.items():
    ax.plot(monthly["Month"], monthly[col], label=col, color=color, linewidth=2)

ax.axvline(CHATGPT_LAUNCH, color="#B44", linestyle="--", linewidth=1.2, alpha=0.8)
ax.text(CHATGPT_LAUNCH, ax.get_ylim()[1]*0.95, "  ChatGPT launch", color="#B44", fontsize=9, va="top")

ax.set_title("Worldwide Google Trends search interest, 2016–2026", fontsize=14, weight="bold")
ax.set_ylabel("Relative search interest (0–100)")
ax.legend(frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.savefig("headline_trend.png", dpi=150)
plt.show()

## 3. When did ChatGPT overtake Google Translate?

Find the first month where ChatGPT's search interest exceeds Google Translate's.

In [ ]:
crossover = monthly[monthly["ChatGPT"] > monthly["Google Translate"]].sort_values("Month").head(1)
if not crossover.empty:
    date = crossover["Month"].dt.strftime("%B %Y").values[0]
    print(f"Crossover month: {date}")
    print(crossover[["Month", "Google Translate", "ChatGPT"]].to_string(index=False))
else:
    print("No crossover found in this window.")

## 4. Segmented trend: did Google Translate's trajectory change after ChatGPT launched?

Fit separate linear trends to Google Translate's monthly search interest before and after November 2022.

In [ ]:
def segmented_trend(df, col, cutoff=CHATGPT_LAUNCH):
    df = df.sort_values("Month").reset_index(drop=True)
    df["t"] = np.arange(len(df))
    pre = df[df["Month"] < cutoff]
    post = df[df["Month"] >= cutoff]

    results = {}
    for label, seg in [("pre_chatgpt", pre), ("post_chatgpt", post)]:
        slope, intercept = np.polyfit(seg["t"], seg[col], 1)
        results[label] = {"slope_per_month": round(slope, 3), "n_months": len(seg)}
    return results

gt_trend = segmented_trend(monthly, "Google Translate")
for period, stats in gt_trend.items():
    direction = "declining" if stats["slope_per_month"] < 0 else "growing"
    print(f"{period}: {direction} at {stats['slope_per_month']} index points/month (n={stats['n_months']} months)")

## 5. Cross-correlation: does ChatGPT's rise predict Google Translate's decline?

Normalize both series (z-score) and test correlation across a ±6 month lag window.

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std()

m = monthly.copy()
m["gt_z"] = zscore(m["Google Translate"])
m["chatgpt_z"] = zscore(m["ChatGPT"])

max_lag = 6
lags = range(-max_lag, max_lag + 1)
corrs = [m["gt_z"].corr(m["chatgpt_z"].shift(lag)) for lag in lags]
xcorr = pd.DataFrame({"lag_months": list(lags), "correlation": corrs})

best = xcorr.loc[xcorr["correlation"].idxmin()]
print(f"Strongest (most negative) correlation: r = {best['correlation']:.3f} at lag {int(best['lag_months'])} months")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(xcorr["lag_months"], xcorr["correlation"], color="#3F8C79")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Lag (months) — negative = ChatGPT leads")
ax.set_ylabel("Correlation")
ax.set_title("Time-lagged cross-correlation: ChatGPT vs. Google Translate", fontsize=12, weight="bold")
plt.tight_layout()
plt.savefig("cross_correlation.png", dpi=150)
plt.show()

## 6. Relative share of attention

What fraction of *combined* search interest across all three tools does each one hold, month by month?

In [ ]:
share = monthly.set_index("Month")[["Google Translate", "ChatGPT", "DeepL"]]
share = share.div(share.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.stackplot(share.index, share["Google Translate"], share["ChatGPT"], share["DeepL"],
             labels=["Google Translate", "ChatGPT", "DeepL"],
             colors=["#C89B3C", "#3F8C79", "#8A8370"], alpha=0.9)
ax.axvline(CHATGPT_LAUNCH, color="white", linestyle="--", linewidth=1.2)
ax.set_title("Relative share of combined search attention, 2016–2026", fontsize=14, weight="bold")
ax.set_ylabel("Share of combined search interest")
ax.legend(loc="upper left", frameon=False)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("relative_share_notebook.png", dpi=150)
plt.show()

print("Most recent month's share:")
print((share.iloc[-1] * 100).round(1).astype(str) + "%")

## 7. Annotated timeline: search interest vs. real product milestones

Does search interest move around actual launch dates, or does it drift independently of them?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for col, color in colors.items():
    ax.plot(monthly["Month"], monthly[col], label=col, color=color, linewidth=1.8)

y_top = monthly[["Google Translate", "ChatGPT", "DeepL"]].values.max()
for i, row in milestones.sort_values("date").iterrows():
    ax.axvline(row["date"], color="grey", linestyle=":", linewidth=0.8, alpha=0.6)
    ax.annotate(row["event"], xy=(row["date"], y_top),
                xytext=(row["date"], y_top + 4 + (i % 3) * 8),
                fontsize=7.5, rotation=0, ha="left", color="#444",
                arrowprops=dict(arrowstyle="-", color="grey", alpha=0.5, lw=0.6))

ax.set_title("Search interest annotated with product milestones", fontsize=14, weight="bold")
ax.set_ylabel("Relative search interest (0–100)")
ax.legend(loc="upper left", frameon=False)
ax.set_ylim(0, y_top + 40)
plt.tight_layout()
plt.savefig("milestones_annotated.png", dpi=150)
plt.show()

## Takeaways

- ChatGPT's search interest first overtook Google Translate's globally in **February 2024**, about 15 months after launch, and has stayed higher in most months since.
- Google Translate's search trend shows a **visible slope change** after ChatGPT's November 2022 launch - not a collapse, but a sustained gradual decline from its pre-2022 range.
- The cross-correlation analysis shows ChatGPT's growth **leads** Google Translate's decline by a few months, consistent with a substitution effect rather than pure coincidence.
- DeepL's line stays flat and small throughout - this shift in attention appears to be specific to ChatGPT, not "AI tools" generally.

**Caveat:** Google Trends measures *search* interest, i.e. discovery/curiosity queries - not actual usage. Google Translate still serves well over a billion monthly active users through direct app/browser integrations that never touch a search bar, so this is a signal about public *attention*, not a claim about total translation volume.

---

### Related work
-  **Full report & methodology (regression tables, limitations section):** [GitHub repo](https://github.com/rahman41i/An-Empirical-Analysis-of-Google-Translate-vs.-ChatGPT)
-  **This dataset:** if this notebook was useful, an upvote on the dataset helps other people find it too.

Feedback and forks welcome - curious what this looks like if you add Gemini or Claude/Perplexity search trends to the comparison.